<div align="right">&copy; Guven</div>

# Module 8 — Generative Agents and Cognitive Modeling

Module 6 gave you a checkpointer — memory *within* a thread. Module 7 showed agents that must model each other. Today: memory __across__ threads, and the three operations that turn a log into a mind — __retrieve, reflect, plan.__

Runs offline. No API key, no embedding service, no vector database.

__Question:__ what should the agent remember, and what should it put in the prompt right now? Those are different questions, and the second one is the hard one.

In [1]:
import math, random, re
from dataclasses import dataclass, field
from typing import List, Dict

def tokens(s: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", s.lower())

def cosine(a: str, b: str) -> float:
    """Bag-of-words cosine. A real system uses embeddings; the logic is identical."""
    ta, tb = tokens(a), tokens(b)
    if not ta or not tb:
        return 0.0
    va, vb = {}, {}
    for t in ta: va[t] = va.get(t, 0) + 1
    for t in tb: vb[t] = vb.get(t, 0) + 1
    dot = sum(va[t] * vb.get(t, 0) for t in va)
    na = math.sqrt(sum(v * v for v in va.values()))
    nb_ = math.sqrt(sum(v * v for v in vb.values()))
    return dot / (na * nb_) if na and nb_ else 0.0

print("ready")

ready


---
## Part A — The baseline, first

Module 7's rule: you cannot claim an architecture helped without a baseline you actually ran. That applies to memory too.

A travel-support agent talks to the same user across many sessions. Some questions can only be answered from things said *earlier*.

In [2]:
HISTORY = [
    # (timestep, what the agent observed)
    (1,  "User said they are vegetarian and dislike spicy food."),
    (2,  "User booked AA210 to Boston, aisle seat, 380 dollars."),
    (3,  "User complained the hotel gym was closed."),
    (4,  "User said their company caps airfare at 400 dollars."),
    (5,  "User asked about vegetarian restaurants near the conference centre."),
    (6,  "User mentioned they get motion sick on small planes."),
    (7,  "User's manager is called Priya and approves all travel."),
    (8,  "User prefers morning flights, never before 07:00."),
    (9,  "User said the Hyatt was fine but overpriced at 210 a night."),
    (10, "User asked to avoid connections through Chicago in winter."),
    (11, "User rated the Boston trip 4 out of 5 overall."),
    (12, "User is travelling to Denver next month for a client visit."),
]

QUESTIONS = [
    ("Book me a flight to Denver.",                    {4, 6, 8, 10, 12}),
    ("Where should we eat after the client meeting?",  {1, 5, 12}),
    ("Is a 430 dollar fare acceptable?",               {4, 7}),
    ("Which hotel should I use this time?",            {3, 9}),
    ("Any airports I should avoid?",                   {10, 6}),
]

def answer_without_memory(q):
    return set()          # knows nothing beyond this turn

def answer_with_all_memory(q):
    return {t for t, _ in HISTORY}      # dumps the entire history into the prompt

def score(retrieved, needed):
    """Recall of the facts that mattered, and how much irrelevant material came along."""
    hit = len(retrieved & needed) / len(needed)
    noise = len(retrieved - needed) / max(1, len(retrieved))
    return hit, noise

print(f"{'question':<48}{'recall':>8}{'noise':>8}{'items':>7}")
print("-" * 71)
for label, fn in [("NO MEMORY", answer_without_memory), ("ALL MEMORY", answer_with_all_memory)]:
    print(f"\n{label}")
    for q, needed in QUESTIONS:
        r = fn(q)
        hit, noise = score(r, needed)
        print(f"  {q:<46}{hit:>8.0%}{noise:>8.0%}{len(r):>7}")

question                                          recall   noise  items
-----------------------------------------------------------------------

NO MEMORY
  Book me a flight to Denver.                         0%      0%      0
  Where should we eat after the client meeting?       0%      0%      0
  Is a 430 dollar fare acceptable?                    0%      0%      0
  Which hotel should I use this time?                 0%      0%      0
  Any airports I should avoid?                        0%      0%      0

ALL MEMORY
  Book me a flight to Denver.                       100%     58%     12
  Where should we eat after the client meeting?     100%     75%     12
  Is a 430 dollar fare acceptable?                  100%     83%     12
  Which hotel should I use this time?               100%     83%     12
  Any airports I should avoid?                      100%     83%     12


> Neither of these is a memory system. One remembers nothing; the other remembers everything
> and pays for it on every single request — in tokens, in latency, and (Part C) in accuracy.
>
> The interesting engineering is entirely in between.

---
## Part B — The memory stream

Park et al.'s generative agents score every memory on three axes and retrieve the top few:

$\mathrm{score}(m) = \alpha_r\,\mathrm{rec}(m) + \alpha_i\,\mathrm{imp}(m) + \alpha_v\,\mathrm{rel}(m, q)$

| | |
|---|---|
| __recency__ | how long ago, decayed: $\mathrm{rec}(m) = \gamma^{\,t - t_m}$ |
| __importance__ | how significant the memory is, scored once when stored |
| __relevance__ | similarity to the current query |

__Relevance alone is not enough__ — it retrieves the same three memories forever and never surfaces something recent. That is why there are three terms.

In [3]:
@dataclass
class Memory:
    t: int
    text: str
    importance: float = 0.5

IMPORTANCE = {1: 0.7, 2: 0.4, 3: 0.3, 4: 0.9, 5: 0.5,
              6: 0.8, 7: 0.6, 8: 0.7, 9: 0.4, 10: 0.7, 11: 0.2, 12: 0.8}

STREAM = [Memory(t, txt, IMPORTANCE[t]) for t, txt in HISTORY]
NOW = 13
GAMMA = 0.90

def retrieve(query, k=4, w_rec=1.0, w_imp=1.0, w_rel=1.0, stream=STREAM, now=NOW):
    scored = []
    for m in stream:
        rec = GAMMA ** (now - m.t)
        rel = cosine(m.text, query)
        s = w_rec * rec + w_imp * m.importance + w_rel * rel
        scored.append((s, rec, m.importance, rel, m))
    scored.sort(key=lambda x: -x[0])
    return scored[:k]

q = "Book me a flight to Denver."
print(f'query: "{q}"\n')
print(f"{'score':>7}{'rec':>7}{'imp':>7}{'rel':>7}  memory")
print("-" * 78)
for s, rec, imp, rel, m in retrieve(q, k=6):
    print(f"{s:>7.2f}{rec:>7.2f}{imp:>7.2f}{rel:>7.2f}  [{m.t:>2}] {m.text[:44]}")

query: "Book me a flight to Denver."

  score    rec    imp    rel  memory
------------------------------------------------------------------------------
   2.07   0.90   0.80   0.37  [12] User is travelling to Denver next month for 
   1.57   0.73   0.70   0.14  [10] User asked to avoid connections through Chic
   1.29   0.59   0.70   0.00  [ 8] User prefers morning flights, never before 0
   1.29   0.39   0.90   0.00  [ 4] User said their company caps airfare at 400 
   1.28   0.48   0.80   0.00  [ 6] User mentioned they get motion sick on small
   1.17   0.66   0.40   0.12  [ 9] User said the Hyatt was fine but overpriced 


### ✏️ Exercise B — the weights are the design

Set two of the three weights to `0` and see what each term is actually doing.

- `w_rel=0` → what does it retrieve? (Whatever is recent and important — regardless of the question)
- `w_rec=0, w_imp=0` → pure similarity search. This is what a naive RAG setup does.
- What breaks in a long-running agent when `w_rec=0`?

In [4]:
for label, kw in [("recency only", dict(w_imp=0, w_rel=0)),
                  ("importance only", dict(w_rec=0, w_rel=0)),
                  ("relevance only (naive RAG)", dict(w_rec=0, w_imp=0)),
                  ("all three", dict())]:
    got = {m.t for *_, m in retrieve(q, k=4, **kw)}
    needed = dict(QUESTIONS)[q]
    hit, noise = score(got, needed)
    print(f"{label:<28} retrieved {sorted(got)}  recall {hit:.0%}  noise {noise:.0%}")

recency only                 retrieved [9, 10, 11, 12]  recall 40%  noise 50%
importance only              retrieved [1, 4, 6, 12]  recall 60%  noise 25%
relevance only (naive RAG)   retrieved [2, 9, 10, 12]  recall 40%  noise 50%
all three                    retrieved [4, 8, 10, 12]  recall 80%  noise 0%


---
## Part C — More memory is not better

This is the Module 2 idea again. Value of information said: gather information only when it changes what you do. Retrieval is an information-gathering action with a price.

Every retrieved memory costs tokens __and__ competes for the model's attention. Below, a model of that trade: recall rises with $k$, but irrelevant context degrades the answer.

In [5]:
TOKENS_PER_MEMORY = 90
BASE_ACCURACY = 0.55            # what the agent gets right with no memory at all

DISTRACTION = 0.055        # cost per irrelevant item sitting in the context

def answer_quality(recall, n_irrelevant):
    """Recall helps. Each irrelevant item in context costs a fixed amount of attention."""
    return max(0.0, min(1.0, BASE_ACCURACY + 0.45 * recall - DISTRACTION * n_irrelevant))

def sweep(stream=STREAM, ks=range(0, 13)):
    rows = []
    for k in ks:
        hits = junk = 0.0
        for qq, needed in QUESTIONS:
            got = {m.t for *_, m in retrieve(qq, k=k, stream=stream)} if k else set()
            hits += len(got & needed) / len(needed) / len(QUESTIONS)
            junk += len(got - needed) / len(QUESTIONS)
        rows.append((k, hits, junk, answer_quality(hits, junk)))
    return rows

print(f"{'k':>3}{'recall':>9}{'junk items':>12}{'quality':>10}{'tokens':>9}")
print("-" * 45)
rows = sweep()
for k, hits, junk, qual in rows:
    print(f"{k:>3}{hits:>9.0%}{junk:>12.1f}{qual:>10.2f}{k*TOKENS_PER_MEMORY:>9}")

best = max(rows, key=lambda r: r[3])
worst_all = rows[-1]
print(f"\nBest k = {best[0]} at quality {best[3]:.2f}")
print(f"Dumping all 12 memories: quality {worst_all[3]:.2f}, "
      f"{12*TOKENS_PER_MEMORY} tokens per request.")
print(f"That is {best[3]-worst_all[3]:+.2f} quality for "
      f"{(12-best[0])*TOKENS_PER_MEMORY} extra tokens -- worse, and dearer.")
print("'Just put it all in the context window' is not a strategy.")

  k   recall  junk items   quality   tokens
---------------------------------------------
  0       0%         0.0      0.55        0
  1      11%         0.6      0.57       90
  2      25%         1.2      0.59      180
  3      29%         2.0      0.57      270
  4      43%         2.6      0.60      360
  5      57%         3.2      0.63      450
  6      67%         4.0      0.63      540
  7      77%         4.8      0.63      630
  8      77%         5.8      0.58      720
  9      83%         6.6      0.56      810
 10      90%         7.4      0.55      900
 11     100%         8.2      0.55      990
 12     100%         9.2      0.49     1080

Best k = 7 at quality 0.63
Dumping all 12 memories: quality 0.49, 1080 tokens per request.
That is +0.14 quality for 450 extra tokens -- worse, and dearer.
'Just put it all in the context window' is not a strategy.


> Context is a budget, not a bucket.
>
> $\sum_{m \in M_k} \mathrm{len}(m) \le B$ — and the binding constraint is usually attention,
> not the token limit.

### ✏️ Exercise C
Raise `BASE_ACCURACY` to 0.85 (a stronger model). Does the optimal $k$ go up or down?
What does that tell you about memory systems built for last year's models?

In [6]:
# TODO: vary BASE_ACCURACY and the noise penalty. Where does optimal k move?

---
## Part D — Reflection: memories about memories

Raw observations are low-level and repetitive. __Reflection__ periodically reads a batch of them and writes a *higher-level* memory back into the stream:

$r = f(\{m_1, \ldots, m_k\})$

The reflection is stored like any other memory, so it can be retrieved later — and can itself be reflected on. That recursion is what the Park et al. paper means by an agent that "understands itself over time."

Our `reflect()` is rule-based so it runs offline. A real one is a single model call:
"here are 20 observations; what are three higher-level things that are true?"

In [7]:
def reflect(stream, now):
    """Synthesise higher-level memories from patterns in the raw stream."""
    text = " ".join(m.text.lower() for m in stream)
    out = []
    if "vegetarian" in text and "spicy" in text:
        out.append(Memory(now, "User has dietary constraints: vegetarian, avoids spicy food.", 0.8))
    if "caps airfare" in text and "overpriced" in text:
        out.append(Memory(now, "User is cost-sensitive and bound by a company travel cap.", 0.9))
    if "motion sick" in text and "avoid connections" in text:
        out.append(Memory(now, "User has strong constraints on aircraft type and routing.", 0.9))
    if "morning flights" in text and "never before" in text:
        out.append(Memory(now, "User has a narrow acceptable departure window in the morning.", 0.7))
    if "no restrictions" in text:
        out.append(Memory(now, "User is flexible about aircraft type and routing; "
                                "no restrictions apply.", 0.9))
    return out

REFLECTIONS = reflect(STREAM, NOW)
print("reflections written back into the stream:\n")
for r in REFLECTIONS:
    print(f"  [{r.t}] (imp {r.importance}) {r.text}")

STREAM2 = STREAM + REFLECTIONS

def covers(m, needed):
    """Which needed observations does this retrieved item account for?
    A raw memory covers itself. A reflection covers the observations it summarised."""
    if m.t <= 12:
        return {m.t} & needed
    words = set(tokens(m.text))
    return {t for t, txt in HISTORY
            if t in needed and len(words & set(tokens(txt))) >= 2}

K = 4
print(f"\nboth retrieving k={K} -- same number of slots in the prompt\n")
print(f"{'':<22}{'recall':>9}{'wasted slots':>14}{'quality':>10}")
print("-" * 56)
for label, st in [("raw stream only", STREAM), ("raw + reflections", STREAM2)]:
    hits = wasted = 0.0
    for qq, needed in QUESTIONS:
        got = retrieve(qq, k=K, stream=st)
        covered, useless = set(), 0
        for *_, m in got:
            c = covers(m, needed)
            covered |= c
            useless += (len(c) == 0)          # this slot earned nothing
        hits += len(covered) / len(needed) / len(QUESTIONS)
        wasted += useless / len(QUESTIONS)
    print(f"{label:<22}{hits:>9.0%}{wasted:>14.1f}{answer_quality(hits, wasted):>10.2f}")

print("\nReflection is COMPRESSION. Four higher-level memories carry what a dozen raw")
print("observations said, and they fit in the same k. This is how an agent keeps a long")
print("history usable without a bigger context window.")

reflections written back into the stream:

  [13] (imp 0.8) User has dietary constraints: vegetarian, avoids spicy food.
  [13] (imp 0.9) User is cost-sensitive and bound by a company travel cap.
  [13] (imp 0.9) User has strong constraints on aircraft type and routing.
  [13] (imp 0.7) User has a narrow acceptable departure window in the morning.

both retrieving k=4 -- same number of slots in the prompt

                         recall  wasted slots   quality
--------------------------------------------------------
raw stream only             43%           2.6      0.60
raw + reflections           80%           1.6      0.82

Reflection is COMPRESSION. Four higher-level memories carry what a dozen raw
observations said, and they fit in the same k. This is how an agent keeps a long
history usable without a bigger context window.


---
## Part E — Memory poisoning

Module 5 showed one wrong fact propagating within a session. Module 7 showed an error becoming consensus across agents. __Memory makes it permanent.__

A wrong memory is retrieved again and again, gets reflected into a higher-level belief, and the belief outlives the observation that caused it.

In [8]:
POISON = Memory(6, "User said they are happy to fly any aircraft, no restrictions.", 0.8)

BAD_PHRASES = ("no restrictions", "flexible about aircraft")

def run_with(stream, label):
    """How often does the bad belief -- or anything derived from it -- reach the prompt?"""
    reached = 0
    for qq, _ in QUESTIONS:
        got = {m.text.lower() for *_, m in retrieve(qq, k=4, stream=stream)}
        reached += any(any(b in t for b in BAD_PHRASES) for t in got)
    print(f"{label:<40}{reached}/{len(QUESTIONS)} prompts contaminated")

clean = STREAM
poisoned = [m for m in STREAM if m.t != 6] + [POISON]
poisoned_reflected = poisoned + reflect(poisoned, NOW)

run_with(clean, "clean stream")
run_with(poisoned, "one wrong memory")
run_with(poisoned_reflected, "wrong memory, then reflected")

print("\nNote what reflection did. It read the contradiction-free (wrong) stream and wrote a")
print("HIGHER-LEVEL belief on top of it. Now the error has a confident summary defending it,")
print("and deleting the original observation no longer removes it.")
print("\nThis is why production memory systems store PROVENANCE: which observation,")
print("from when, and how confident. Module 5's calibration problem, with a longer memory.")

clean stream                            0/5 prompts contaminated
one wrong memory                        2/5 prompts contaminated
wrong memory, then reflected            5/5 prompts contaminated

Note what reflection did. It read the contradiction-free (wrong) stream and wrote a
HIGHER-LEVEL belief on top of it. Now the error has a confident summary defending it,
and deleting the original observation no longer removes it.

This is why production memory systems store PROVENANCE: which observation,
from when, and how confident. Module 5's calibration problem, with a longer memory.


### ✏️ Exercise E
Add a `source` and `confidence` field to `Memory`. What retrieval rule would let a later, better-sourced observation override an earlier one?

> There is no clean answer here. Belief revision is genuinely unsolved, and every memory
> system you will use has made a rough choice about it.

---
## Part F — Procedural memory: skills

Three kinds of long-term memory, and agent frameworks have names for all three:

| Cognitive term | What it holds | In an agent |
|---|---|---|
| __Episodic__ | what happened | the memory stream above |
| __Semantic__ | facts about the world | a vector store / knowledge base |
| __Procedural__ | how to do things | __skills__ — reusable, loaded on demand |

A skill is a procedure the agent worked out once and can reload. It is memory that __executes__.

In [9]:
SKILLS = {}

def learn_skill(name, trigger, steps):
    SKILLS[name] = {"trigger": trigger, "steps": steps, "used": 0}

def find_skill(task):
    best, best_sim = None, 0.0
    for name, s in SKILLS.items():
        sim = cosine(s["trigger"], task)
        if sim > best_sim:
            best, best_sim = name, sim
    return (best, best_sim) if best_sim > 0.35 else (None, best_sim)

# The agent solves a task the slow way once, then writes down what worked.
learn_skill("book_compliant_flight",
            "book a flight within company policy",
            ["check policy cap", "search flights under cap", "filter by departure window",
             "exclude disallowed connections", "propose top 2 to user"])
learn_skill("find_dining",
            "where should we eat near the meeting venue restaurants",
            ["read dietary constraints from memory", "search near venue",
             "filter by constraints", "rank by rating"])

TASKS = ["book a flight to Denver within policy",
         "where should we eat after the client meeting",
         "file an expense report"]

STEPS_FROM_SCRATCH = 9
for task in TASKS:
    name, sim = find_skill(task)
    if name:
        SKILLS[name]["used"] += 1
        print(f'"{task[:44]:<46}" -> skill {name} (sim {sim:.2f}), '
              f'{len(SKILLS[name]["steps"])} steps')
    else:
        print(f'"{task[:44]:<46}" -> no skill, plan from scratch, '
              f'~{STEPS_FROM_SCRATCH} steps')

reused = sum(s["used"] for s in SKILLS.values())
print(f"\n{reused}/{len(TASKS)} tasks solved from procedural memory.")
print("A skill is a plan the agent does not have to re-derive -- and, unlike a prompt,")
print("it is inspectable, versionable, and reviewable before it runs.")

"book a flight to Denver within policy         " -> skill book_compliant_flight (sim 0.77), 5 steps
"where should we eat after the client meeting  " -> skill find_dining (sim 0.71), 4 steps
"file an expense report                        " -> no skill, plan from scratch, ~9 steps

2/3 tasks solved from procedural memory.
A skill is a plan the agent does not have to re-derive -- and, unlike a prompt,
it is inspectable, versionable, and reviewable before it runs.


> This is where Module 9's material begins. Skills are loaded into a harness that decides
> what the agent may execute and in what sandbox. Memory that executes is memory that needs a
> security boundary.

---
## &#128218; References
1. Russell, Stuart, and Peter Norvig. Artificial Intelligence: A Modern Approach. 4th ed., Pearson, 2020.
2. LangChain. "LangGraph Overview." LangChain Docs, 2026, docs.langchain.com/oss/python/langgraph/overview. Accessed Aug 2026

---
## &#127939; Exercises
__Exercise 1.__ Finish all the exercises above.

---

In [10]:
%%html
<style>
    table {margin-left: 0 !important;}
    p {font-family: verdana;}
    li {font-family: verdana;}
    div {font-size: 10pt;}
</style>
<!-- Display markdown tables left oriented in this notebook. -->

---
---